# CSET485 – AI and Society – Assignment 1
**Roll no. ending 0028 → Adult Census Income dataset, `random_state = 28`, 70/30 split.**

Run top to bottom. Every number in the PDF comes from this notebook. All results are also saved to `results.json`, tables to `outputs/`.

In [ ]:
# !pip install statsmodels scikit-learn pandas numpy matplotlib scipy
import json, os, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, accuracy_score
warnings.filterwarnings("ignore")

SEED = 28                      # last 3 digits of roll no. (028)
os.makedirs("outputs", exist_ok=True)
R = {}                         # every reported number is stored here -> results.json
pd.set_option("display.width", 200, "display.max_columns", 50)

## Data loading

In [ ]:
URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
cols = ['age','workclass','fnlwgt','education','education-num','marital-status','occupation',
        'relationship','race','sex','capital-gain','capital-loss','hours-per-week','native-country','income']
# If the URL is blocked, download adult.data manually and use the local path instead.
df = pd.read_csv(URL, names=cols, skipinitialspace=True, na_values='?')
print(df.shape)
df.head()

## Data preprocessing (documented as required)
* **Missing values:** `?` appears in `workclass`, `occupation`, `native-country`. These are categorical and missingness is informative (likely not random), so they are kept as an explicit **"Unknown"** category rather than dropped (dropping would lose rows and could bias the sample).
* **Duplicates:** exact duplicate rows are dropped.
* **Dropped columns:** `fnlwgt` (census sampling weight, not a person attribute); `education` (redundant with ordinal `education-num`).
* **Categorical encoding:** one-hot with `drop_first=True` (avoids perfect collinearity for OLS/Logit). `native-country` collapsed to United-States vs Other; `race` collapsed to White / Black / Other because of sparse categories.
* **Outliers:** `capital-gain` / `capital-loss` are extremely skewed (99999 is a top-code) → `log1p` transform. `age` and `hours-per-week` are winsorised at the 1st/99th percentile **computed on the training set only**.
* **Scaling:** numeric columns standardised (z-score) with a scaler **fit on train only** (no leakage). Coefficients are therefore per 1 SD for numeric predictors.
* **Target:** `income` = 1 if >50K else 0.

In [ ]:
print("Missing values (before):"); print(df.isna().sum()[lambda s: s>0])
n0 = len(df)
df = df.drop_duplicates().reset_index(drop=True)
R['rows_raw']=n0; R['rows_after_dedup']=len(df)
for c in ['workclass','occupation','native-country']:
    df[c] = df[c].fillna('Unknown')

y = (df['income']=='>50K').astype(int)
R['positive_rate_overall'] = float(y.mean())

X = df.drop(columns=['fnlwgt','education','income']).copy()
X['native-country'] = np.where(X['native-country']=='United-States','US','Other')
X['race'] = X['race'].where(X['race'].isin(['White','Black']), 'Other')
X['capital-gain'] = np.log1p(X['capital-gain'])
X['capital-loss'] = np.log1p(X['capital-loss'])
cat_cols = X.select_dtypes('object').columns.tolist()
num_cols = ['age','education-num','capital-gain','capital-loss','hours-per-week']
X = pd.get_dummies(X, columns=cat_cols, drop_first=True).astype(float)
X.columns = [c.replace('-','_').replace(' ','_').replace('&','and') for c in X.columns]
num_cols = [c.replace('-','_') for c in num_cols]

# ---- split FIRST, then fit caps/scaler on train only ----
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.30, random_state=SEED)
for c in ['age','hours_per_week']:
    lo, hi = X_tr[c].quantile([.01,.99])
    X_tr[c] = X_tr[c].clip(lo,hi); X_te[c] = X_te[c].clip(lo,hi)
sc = StandardScaler().fit(X_tr[num_cols])
X_tr[num_cols] = sc.transform(X_tr[num_cols]); X_te[num_cols] = sc.transform(X_te[num_cols])
R['n_train'], R['n_test'] = len(X_tr), len(X_te)
print(X_tr.shape, X_te.shape, "positive rate train/test:", y_tr.mean().round(4), y_te.mean().round(4))

## Part 1 – Build: Model A (OLS on 0/1) vs Model B (Logistic)

In [ ]:
Xtr_c, Xte_c = sm.add_constant(X_tr), sm.add_constant(X_te)
ols   = sm.OLS(y_tr, Xtr_c).fit()
logit = sm.Logit(y_tr, Xtr_c).fit(disp=0, maxiter=300)

pA = ols.predict(Xte_c)                 # may fall outside [0,1] – itself a finding
pB = logit.predict(Xte_c)
med = float(np.median(pA))
highA = (pA >= med).astype(int)         # median threshold
highB = (pB >= 0.5).astype(int)         # 0.5 threshold

cm = pd.crosstab(pd.Series(highA,name='Model A (median thr)',index=X_te.index),
                 pd.Series(highB,name='Model B (p>=0.5)',index=X_te.index))
print(cm)
mis = int((highA!=highB).sum())
R['part1'] = dict(median_ols_pred=med, ols_pred_min=float(pA.min()), ols_pred_max=float(pA.max()),
    n_ols_outside_01=int(((pA<0)|(pA>1)).sum()),
    pct_flagged_high_A=float(highA.mean()), pct_flagged_high_B=float(highB.mean()),
    confusion=cm.values.tolist(), mismatches=mis, mismatch_pct=float(mis/len(y_te)),
    A_only=int(((highA==1)&(highB==0)).sum()), B_only=int(((highA==0)&(highB==1)).sum()),
    acc_A=float(accuracy_score(y_te,highA)), acc_B=float(accuracy_score(y_te,highB)),
    ols_r2_train=float(ols.rsquared))
print(json.dumps(R['part1'], indent=1))

**Why they differ (for the report):** the median rule *forces* 50% of people into "high risk" regardless of calibration, while the logistic 0.5 rule flags only those whose predicted probability is ≥ 0.5 – with a ~24% base rate this is a much smaller group. The median threshold is a *rank* cut, 0.5 is a *probability* cut; OLS also ignores the bounded 0–1 / sigmoid shape (some predictions fall outside [0,1]).

## 2.1 – Simpson's paradox hunt (sex, race, age subgroups)

In [ ]:
X_all = pd.concat([X_tr, X_te]).sort_index(); y_all = y.loc[X_all.index]

def fit_logit(Xd, yd):
    Xd = Xd.loc[:, Xd.std()>0]                       # drop constant columns inside a subgroup
    m = sm.Logit(yd, sm.add_constant(Xd)).fit(disp=0, maxiter=300)
    return m

full = fit_logit(X_all, y_all)
splits = {'sex': ('sex_Male', X_all['sex_Male']>0),
          'race_White': ('race_White', X_all.get('race_White', pd.Series(0,index=X_all.index))>0),
          'age>=median': ('age', X_all['age']>=X_all['age'].median())}
# race_White doesn't exist if dropped as the baseline -> use ~race_Black fallback
if 'race_White' not in X_all.columns:
    splits['race_White'] = ('race_Black', X_all['race_Black']==0)

rows=[]
for name,(_,mask) in splits.items():
    mA, mB = fit_logit(X_all[mask], y_all[mask]), fit_logit(X_all[~mask], y_all[~mask])
    for p in num_cols + [c for c in X_all.columns if c not in num_cols]:
        if p in mA.params and p in mB.params and p in full.params:
            ciA, ciB = mA.conf_int().loc[p], mB.conf_int().loc[p]
            sA, sB = mA.bse[p], mB.bse[p]
            z = (mA.params[p]-mB.params[p])/np.sqrt(sA**2+sB**2)
            rows.append(dict(split=name, predictor=p, full=full.params[p], A=mA.params[p], B=mB.params[p],
                             A_lo=ciA[0],A_hi=ciA[1],B_lo=ciB[0],B_hi=ciB[1], pA=mA.pvalues[p], pB=mB.pvalues[p],
                             z_diff=z, sign_flip=(np.sign(mA.params[p])!=np.sign(mB.params[p])),
                             sig_flip=(np.sign(mA.params[p])!=np.sign(mB.params[p])) and mA.pvalues[p]<.05 and mB.pvalues[p]<.05))
simp = pd.DataFrame(rows)
simp.to_csv("outputs/simpson_scan.csv", index=False)
print("Significant sign reversals (both subgroups p<.05):")
print(simp[simp.sig_flip].round(3).to_string())
print("\nLargest subgroup differences by |z|:")
print(simp.reindex(simp.z_diff.abs().sort_values(ascending=False).index).head(10).round(3).to_string())

In [ ]:
# --- Choose the headline case (edit KEY / SPLIT after looking at the scan above) ---
SPLIT, KEY = 'sex', 'education_num'
_, mask = splits[SPLIT]
mA, mB = fit_logit(X_all[mask], y_all[mask]), fit_logit(X_all[~mask], y_all[~mask])
tab = pd.DataFrame({m:[mod.params[KEY], *mod.conf_int().loc[KEY].tolist(), mod.pvalues[KEY]]
        for m,mod in {'Full':full,f'{SPLIT}: A (mask True)':mA,f'{SPLIT}: B (mask False)':mB}.items()},
        index=['coef','CI_low','CI_high','p']).T
print(tab.round(4))
R['part2_1'] = dict(split=SPLIT, predictor=KEY, table=tab.round(5).to_dict('index'),
                    n_A=int(mask.sum()), n_B=int((~mask).sum()),
                    sig_reversals=simp[simp.sig_flip][['split','predictor','full','A','B']].round(4).to_dict('records'))

## 2.2 – Omitted-variable bias (drop marital status)

In [ ]:
drop_cols = [c for c in X_tr.columns if c.startswith('marital_status')]
red_cols  = [c for c in X_tr.columns if c not in drop_cols]
m_full = sm.Logit(y_tr, sm.add_constant(X_tr)).fit(disp=0, maxiter=300)
m_red  = sm.Logit(y_tr, sm.add_constant(X_tr[red_cols])).fit(disp=0, maxiter=300)
cmp = ['sex_Male','age','education_num','hours_per_week','capital_gain','race_Black']
cmp = [c for c in cmp if c in m_red.params]
ovb = pd.DataFrame({'full':m_full.params[cmp],'reduced':m_red.params[cmp]})
ovb['pct_change'] = (ovb.reduced-ovb.full)/ovb.full.abs()*100
print(ovb.round(4)); print("Largest shift:", ovb.pct_change.abs().idxmax())
R['part2_2'] = dict(dropped=drop_cols, table=ovb.round(5).to_dict('index'),
                    largest_shift=ovb.pct_change.abs().idxmax(),
                    acc_full=float(accuracy_score(y_te,(m_full.predict(sm.add_constant(X_te))>=.5))),
                    acc_reduced=float(accuracy_score(y_te,(m_red.predict(sm.add_constant(X_te[red_cols]))>=.5))))
ovb.to_csv("outputs/omitted_variable.csv")

## 2.3 – Adversarial subset (confident wrong predictions)

In [ ]:
pB = logit.predict(Xte_c)
fp = (pB>=.75)&(y_te==0); fn = (pB<=.25)&(y_te==1)
bad = fp|fn
R['part2_3'] = dict(n_fp=int(fp.sum()), n_fn=int(fn.sum()), n_bad=int(bad.sum()), pct_bad=float(bad.mean()*100),
                    n_confident_pos=int((pB>=.75).sum()), n_confident_neg=int((pB<=.25).sum()))
idx = bad[bad].index
samp = pd.Series(idx).sample(min(50,len(idx)), random_state=SEED).values
S = X_te.loc[samp]; T = X_te
# standardised mean difference of every feature: sample vs full test set
smd = ((S.mean()-T.mean())/T.std()).sort_values(key=abs, ascending=False)
print(R['part2_3']); print(smd.head(10).round(2))
# raw-scale demographics
raw = df.loc[samp]; rawT = df.loc[X_te.index]
demo = pd.DataFrame({'adv50':[raw.age.mean(), (raw.sex=='Male').mean(), (raw.race=='White').mean(),
                              (raw['marital-status']=='Married-civ-spouse').mean(), raw['education-num'].mean(), raw['hours-per-week'].mean(),
                              (raw['capital-gain']>0).mean(), y.loc[samp].mean()],
                     'test':[rawT.age.mean(), (rawT.sex=='Male').mean(), (rawT.race=='White').mean(),
                              (rawT['marital-status']=='Married-civ-spouse').mean(), rawT['education-num'].mean(), rawT['hours-per-week'].mean(),
                              (rawT['capital-gain']>0).mean(), y_te.mean()]},
                    index=['mean_age','share_male','share_white','share_married_civ_spouse','mean_edu_num','mean_hours','share_capgain>0','P(income>50K)'])
print(demo.round(3))
print("FP share in sample:", (fp.loc[samp]).mean().round(3), " FN share:", (fn.loc[samp]).mean().round(3))
R['part2_3'].update(top_smd=smd.head(8).round(3).to_dict(), demographics=demo.round(4).to_dict('index'),
                    sample_fp_share=float(fp.loc[samp].mean()), sample_fn_share=float(fn.loc[samp].mean()),
                    top_logit_coefs=m_full.params.drop('const').sort_values(key=abs,ascending=False).head(8).round(3).to_dict())
demo.to_csv("outputs/adversarial_demographics.csv")

## 2.4 – Fairness trade-off (sensitive variable: sex; Group A = Female, Group B = Male)

In [ ]:
female = (X_te['sex_Male']==0).values; male = ~female
yt = y_te.values; ps = pB.values
def safe(a,b): return np.nan if b==0 else a/b
rows=[]
for t in np.arange(0.05,0.96,0.1):
    yh = (ps>=t).astype(int)
    def stats(m):
        yy, hh = yt[m], yh[m]
        pr = hh.mean()
        tpr = hh[yy==1].mean() if (yy==1).any() else np.nan
        fpr = hh[yy==0].mean() if (yy==0).any() else np.nan
        ppv = yy[hh==1].mean() if (hh==1).any() else np.nan
        return pr,tpr,fpr,ppv
    prA,tprA,fprA,ppvA = stats(female); prB,tprB,fprB,ppvB = stats(male)
    DP = safe(prA,prB); PP = safe(ppvA,ppvB)
    EO = np.nanmin([safe(tprA,tprB), safe(fprA,fprB)])
    rows.append(dict(t=round(t,2),DP=DP,EO=EO,PP=PP,acc=(yh==yt).mean(),
                     rate_F=prA,rate_M=prB,TPR_F=tprA,TPR_M=tprB,FPR_F=fprA,FPR_M=fprB,PPV_F=ppvA,PPV_M=ppvB))
fair = pd.DataFrame(rows); print(fair.round(3).to_string()); fair.to_csv("outputs/fairness_table.csv", index=False)

fig,ax = plt.subplots(figsize=(6.5,6))
ax.plot(fair.DP, fair.EO, '-o', color='tab:blue')
for _,r in fair.iterrows(): ax.annotate(f"{r.t:.2f}", (r.DP,r.EO), textcoords='offset points', xytext=(5,5), fontsize=9)
ax.axvline(1,ls='--',c='grey'); ax.axhline(1,ls='--',c='grey'); ax.scatter([1],[1],marker='*',s=200,c='gold',zorder=5)
ax.set_xlim(0,1.5); ax.set_ylim(0,1.5); ax.set_xlabel("Demographic Parity ratio (F/M)"); ax.set_ylabel("Equalized Odds ratio (min of TPR, FPR ratio)")
ax.set_title("Fairness trade-off across thresholds (label = threshold)"); plt.tight_layout(); plt.savefig("outputs/tradeoff_chart.png", dpi=200); plt.show()

# quantify trade-off between the threshold closest to DP=1 and the one closest to EO=1
f = fair.dropna(subset=['DP','EO']).copy()
a = f.loc[(f.DP-1).abs().idxmin()]; b = f.loc[(f.EO-1).abs().idxmin()]
dp_gain = (a.DP-b.DP); eo_loss=(a.EO-b.EO)
R['part2_4'] = dict(table=fair.round(4).to_dict('records'), t_best_DP=float(a.t), t_best_EO=float(b.t),
    DP_at_bestDP=float(a.DP), DP_at_bestEO=float(b.DP), EO_at_bestDP=float(a.EO), EO_at_bestEO=float(b.EO),
    DP_gain_abs=float(dp_gain), DP_gain_pct=float(dp_gain/abs(b.DP)*100) if b.DP else None,
    EO_change_abs=float(eo_loss), EO_change_pct=float(eo_loss/abs(b.EO)*100) if b.EO else None,
    tradeoff_cost=float((abs(eo_loss/b.EO))/(abs(dp_gain/b.DP))) if b.DP and b.EO and dp_gain else None,
    base_rate_F=float(yt[female].mean()), base_rate_M=float(yt[male].mean()),
    n_F=int(female.sum()), n_M=int(male.sum()))
print(json.dumps({k:v for k,v in R['part2_4'].items() if k!='table'}, indent=1))

## Worked individuals for Part 3 Q6 (pick two real test-set people, one F and one M, with similar features)

In [ ]:
cand = pd.DataFrame({'sex':np.where(female,'F','M'),'p':ps,'y':yt,'age':df.loc[X_te.index,'age'].values,
                     'edu':df.loc[X_te.index,'education-num'].values,'hrs':df.loc[X_te.index,'hours-per-week'].values}, index=X_te.index)
# a qualified (y=1) woman and man with similar profile but different scores
w = cand[(cand.sex=='F')&(cand.y==1)].sort_values('p').iloc[len(cand[(cand.sex=='F')&(cand.y==1)])//4]
m = cand[(cand.sex=='M')&(cand.y==1)&(cand.edu==w.edu)].iloc[(cand[(cand.sex=='M')&(cand.y==1)&(cand.edu==w.edu)].p - w.p).abs().argsort().iloc[:1]]
print(w, "\n", m)
ex = {str(t):{'F':bool(w.p>=t),'M':bool(m.p.iloc[0]>=t)} for t in fair.t}
R['q6_example']=dict(F=w.to_dict(), M=m.iloc[0].to_dict(), outcomes_by_threshold=ex)

## Export

In [ ]:
def clean(o):
    if isinstance(o,dict): return {str(k):clean(v) for k,v in o.items()}
    if isinstance(o,(list,tuple)): return [clean(v) for v in o]
    if isinstance(o,(np.floating,)): return None if np.isnan(o) else float(o)
    if isinstance(o,(np.integer,)): return int(o)
    if isinstance(o,float) and np.isnan(o): return None
    return o
json.dump(clean(R), open("results.json","w"), indent=1)
print("Saved results.json and outputs/*")